# TCN — Per-Motor Current Prediction (Power Limiter)

| Property | Value |
|----------|-------|
| **Model** | 4 x MinimalTCN (pure numpy, one per motor) |
| **Input** | [I_FL, I_FR, I_RL, I_RR, T_sum, U_dc] normalized, last 5 ticks |
| **Output** | I_FL, I_FR, I_RL, I_RR at t+1 |
| **Training** | Mini-batch Adam, 50 epochs |

**Architecture per motor:**
```
Input (B, 6, 5)
  -- CausalConv1d(6->16, k=3, d=1) + ReLU
  -- CausalConv1d(16->16, k=3, d=2) + ReLU
  -- Take last timestep -> (B, 16)
  -- Linear(16->1)
```

In [ ]:
import sys, time
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

_cwd = Path().resolve()
SRC_DIR  = _cwd if (_cwd / 'functions').exists() else _cwd / 'src'
DATA_DIR = SRC_DIR.parent / 'data' / 'model'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

DT            = 0.2        # 5 Hz sample rate
N_LAGS        = 5          # 1 s lookback (5 ticks × 0.2 s)
POWER_LIMIT_W = 80_000
MOTOR_NAMES   = ['FL', 'FR', 'RL', 'RR']
from functions.tcn        import MinimalTCN
from functions.evaluation import display_model_results, compare_models

N_SEQ = N_LAGS   # sequence length = 5 ticks
print(f'DATA_DIR : {DATA_DIR}')


## 1. Data Loading

In [ ]:
df = pd.read_csv(DATA_DIR / 'training_data.csv')
df = df.sort_values(['run_id', 'timestamp_s']).reset_index(drop=True)
df['T_sum'] = df['T_FL'] + df['T_FR'] + df['T_RL'] + df['T_RR']

# Run-safe lag features (no cross-run leakage)
for m in MOTOR_NAMES:
    for lag in range(1, N_LAGS + 1):
        df[f'I_{m}_lag{lag}'] = df.groupby('run_id')[f'I_{m}'].shift(lag)
for lag in range(1, N_LAGS + 1):
    df[f'T_sum_lag{lag}'] = df.groupby('run_id')['T_sum'].shift(lag)

# 1-step-ahead target (within each run)
for m in MOTOR_NAMES:
    df[f'I_{m}_next'] = df.groupby('run_id')[f'I_{m}'].shift(-1)

df_clean = df.dropna().reset_index(drop=True)
print(f'Loaded  : {len(df):,} rows, {df["run_id"].nunique()} runs')
print(f'Clean   : {len(df_clean):,} rows after dropna')


In [ ]:
runs         = sorted(df_clean['run_id'].unique())
n_train_runs = max(1, int(len(runs) * 0.70))
train_runs   = runs[:n_train_runs]
test_runs    = runs[n_train_runs:]

df_train = df_clean[df_clean['run_id'].isin(train_runs)].reset_index(drop=True)
df_test  = df_clean[df_clean['run_id'].isin(test_runs)].reset_index(drop=True)

y_train = df_train[[f'I_{m}_next' for m in MOTOR_NAMES]].values   # (n, 4)
y_test  = df_test[[f'I_{m}_next'  for m in MOTOR_NAMES]].values
U_test  = df_test['U_dc'].values
t_test  = df_test['timestamp_s'].values

print(f'Train runs : {train_runs}  ({len(df_train):,} samples)')
print(f'Test  runs : {test_runs}   ({len(df_test):,} samples)')
print(f'y_train    : {y_train.shape}')


## 2. Sequence Building

TCN input: 6-channel sequences of shape (n, 6, 5) — normalized.

In [ ]:
# 6 input channels: I_FL, I_FR, I_RL, I_RR, T_sum, U_dc
ch_cols = ['I_FL', 'I_FR', 'I_RL', 'I_RR', 'T_sum', 'U_dc']
n_ch    = len(ch_cols)

ch_train = df_train[ch_cols].values   # (n_train, 6)
ch_test  = df_test[ch_cols].values    # (n_test,  6)

mu_ch  = ch_train.mean(axis=0)
std_ch = ch_train.std(axis=0) + 1e-8

ch_all  = np.vstack([ch_train, ch_test])
ch_norm = (ch_all - mu_ch) / std_ch

n_train_s = len(ch_train)
n_total   = len(ch_all)

# Build (n, n_ch, N_SEQ) sequences — causal, padded with zeros at run boundaries
# Simple approach: global sequence (slight boundary leakage at run boundaries is acceptable)
X_seq_all = np.zeros((n_total, n_ch, N_SEQ))
for i in range(N_SEQ - 1, n_total):
    X_seq_all[i] = ch_norm[i - N_SEQ + 1 : i + 1].T   # (n_ch, N_SEQ)

X_seq_train = X_seq_all[:n_train_s]
X_seq_test  = X_seq_all[n_train_s:]

y_mu    = y_train.mean(axis=0)
y_sigma = y_train.std(axis=0) + 1e-8
y_train_norm = (y_train - y_mu) / y_sigma

print(f'X_seq_train : {X_seq_train.shape}  (n, channels, timesteps)')
print(f'Channel means : {mu_ch.round(1)}')


## 3. Training — one TCN per motor

In [ ]:
np.random.seed(42)
tcns = {}
hists = {}

for j, m in enumerate(MOTOR_NAMES):
    print(f'--- Motor {m} ---')
    tcn_m = MinimalTCN(n_ch=n_ch, n_filters=16, kernel_size=3, lr=5e-4, seed=42 + j)
    hist  = tcn_m.fit(X_seq_train, y_train_norm[:, j], epochs=50, batch_size=512)
    tcns[m]  = tcn_m
    hists[m] = hist
    print()

fig, axes = plt.subplots(1, 4, figsize=(16, 3))
for ax, m in zip(axes, MOTOR_NAMES):
    ax.plot(hists[m], color='steelblue', lw=1.5)
    ax.set_title(f'Motor {m}'); ax.set_xlabel('Epoch'); ax.set_ylabel('RMSE (norm)')
    ax.grid(True, ls=':', alpha=0.5)
fig.suptitle('TCN training curves', fontsize=12)
plt.tight_layout(); plt.show()


## 4. Evaluation

In [ ]:
t0 = time.perf_counter()
y_pred_norm = np.column_stack([tcns[m].predict(X_seq_test) for m in MOTOR_NAMES])
t_infer = time.perf_counter() - t0

y_pred_tcn = y_pred_norm * y_sigma + y_mu   # denormalize

metrics = display_model_results(
    'TCN 4-motor (numpy)', y_test, y_pred_tcn, t_infer,
    voltage=U_test, motor_names=MOTOR_NAMES, save_to_excel=True,
    history={'train': {m: hists[m] for m in MOTOR_NAMES}},
)


## 5. Time-Series Preview

In [ ]:
N_PLOT = min(200, len(y_test))
sl     = slice(0, N_PLOT)
t_pl   = t_test[sl] - t_test[0]

fig, axes = plt.subplots(4, 1, figsize=(16, 14), sharex=True)
fig.suptitle('TCN — 1-step-ahead per-motor current (5 Hz)', fontsize=13, fontweight='bold')
for i, (ax, m) in enumerate(zip(axes, MOTOR_NAMES)):
    ax.plot(t_pl, y_test[sl, i],     lw=0.9, color='black', alpha=0.6, label='Actual')
    ax.plot(t_pl, y_pred_tcn[sl, i], lw=1.5, color='lime',  ls='--',   label='TCN')
    ax.set_ylabel(f'I_{m} [A]'); ax.legend(fontsize=8); ax.grid(True, ls=':', alpha=0.5)
axes[-1].set_xlabel('Time [s]')
plt.tight_layout(); plt.show()


## Summary

| Property | Value |
|----------|-------|
| Params per motor | ~1 k weights |
| Receptive field | 7 ticks = 1.4 s at 5 Hz |
| Training | Adam, pure numpy |
| C-portable | yes — weight export + matmul |

**Next:** ESN in `06_esn.ipynb`.